<a href="https://colab.research.google.com/github/vibhorvhadke/llm-file-system-assistant/blob/main/Vibhor2_LLM_File_System_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Set up project folder structure
# Creates two folders:
#   - resumes/  -> stores sample resume files (input data)
#   - output/   -> stores generated files like summaries (tool output)

import os

os.makedirs("resumes", exist_ok=True)
os.makedirs("output", exist_ok=True)

print("Folders created:", os.listdir())

Folders created: ['.config', 'llm_file_assistant.py', 'requirements.txt', 'README.md', '__pycache__', 'fs_tools.py', 'resumes', 'output', 'sample_data']


In [ ]:
# Cell 2: Install Python packages needed to read PDF/DOCX files
#          and to generate sample PDF resumes for testing.
#   - pypdf        -> extracts text from PDF files
#   - python-docx  -> extracts text from DOCX (Word) files
#   - reportlab    -> generates a sample PDF resume for testing

!pip install pypdf python-docx reportlab -q

In [ ]:
# Cell 3: Create 6 dummy resume files for testing — a mix of .txt, .docx, and .pdf
#          These act as sample input data for our file system tools.

from docx import Document
from reportlab.pdfgen import canvas

# --- Plain text resumes ---
sample_resumes_txt = {
    "resume_john_doe.txt": """John Doe
Email: john.doe@example.com
Phone: 555-1234

Summary:
Software engineer with 5 years of experience in Python and cloud infrastructure.

Skills:
Python, AWS, Docker, SQL

Experience:
Backend Developer at TechCorp (2020-2025)
- Built REST APIs using Python and Flask
- Automated deployments using AWS
""",
    "resume_jane_smith.txt": """Jane Smith
Email: jane.smith@example.com
Phone: 555-2345

Summary:
Data scientist with 6 years of experience in machine learning and Python.

Skills:
Python, TensorFlow, SQL, Pandas

Experience:
Data Scientist at DataCorp (2019-2025)
- Built ML models for fraud detection using Python
- Worked extensively with SQL databases
""",
    "resume_mike_brown.txt": """Mike Brown
Email: mike.brown@example.com
Phone: 555-3456

Summary:
Frontend developer with 4 years of experience in JavaScript and React.

Skills:
JavaScript, React, CSS, HTML

Experience:
Frontend Developer at WebWorks (2021-2025)
- Built responsive UIs using React
- No Python experience
""",
    "resume_sara_khan.txt": """Sara Khan
Email: sara.khan@example.com
Phone: 555-4567

Summary:
DevOps engineer with 7 years of experience in AWS and Kubernetes.

Skills:
Python, AWS, Kubernetes, Docker

Experience:
DevOps Engineer at CloudNine (2018-2025)
- Automated infrastructure using Python scripts
- Managed AWS and Kubernetes clusters
""",
}

for filename, content in sample_resumes_txt.items():
    with open(f"resumes/{filename}", "w") as f:
        f.write(content)

# --- DOCX resume ---
doc = Document()
doc.add_paragraph("Emily Davis")
doc.add_paragraph("Email: emily.davis@example.com")
doc.add_paragraph("Phone: 555-5678")
doc.add_paragraph("")
doc.add_paragraph("Summary:")
doc.add_paragraph("Backend engineer with 5 years of experience in Python and Java.")
doc.add_paragraph("")
doc.add_paragraph("Skills:")
doc.add_paragraph("Python, Java, Spring Boot, PostgreSQL")
doc.add_paragraph("")
doc.add_paragraph("Experience:")
doc.add_paragraph("Backend Engineer at FinServe (2020-2025)")
doc.add_paragraph("- Built microservices using Python and Spring Boot")
doc.add_paragraph("- Optimized PostgreSQL queries")
doc.save("resumes/resume_emily_davis.docx")

# --- PDF resume ---
c = canvas.Canvas("resumes/resume_raj_patel.pdf")
lines = [
    "Raj Patel",
    "Email: raj.patel@example.com",
    "Phone: 555-6789",
    "",
    "Summary:",
    "Full-stack developer with 3 years of experience in Python and JavaScript.",
    "",
    "Skills:",
    "Python, JavaScript, Django, React",
    "",
    "Experience:",
    "Full-Stack Developer at StartupXYZ (2022-2025)",
    "- Built web apps using Python Django and React",
]
y = 800
for line in lines:
    c.drawString(50, y, line)
    y -= 20
c.save()

print("Created 6 resumes:", os.listdir("resumes"))

Created 6 resumes: ['resume_jane_smith.txt', 'resume_sara_khan.txt', 'resume_emily_davis.docx', 'resume_raj_patel.pdf', 'resume_john_doe.txt', 'resume_mike_brown.txt']


In [ ]:
%%writefile fs_tools.py
# Cell 4: Write fs_tools.py — the core file system tools module.
# This file defines 4 functions the LLM will later call:
#   - read_file       : extract text from a PDF/TXT/DOCX file
#   - list_files      : list files in a directory, optionally filtered by extension
#   - write_file      : write text content to a file (creates folders if needed)
#   - search_in_file   : case-insensitive keyword search with surrounding context

import os
from pypdf import PdfReader
from docx import Document


def read_file(filepath: str) -> dict:
    """
    Read a resume file (PDF, TXT, or DOCX) and extract its text content.
    Returns a structured dict with content and metadata.
    """
    if not os.path.exists(filepath):
        return {"success": False, "error": f"File not found: {filepath}"}

    ext = os.path.splitext(filepath)[1].lower()

    try:
        if ext == ".txt":
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()

        elif ext == ".pdf":
            reader = PdfReader(filepath)
            content = "\n".join(page.extract_text() or "" for page in reader.pages)

        elif ext == ".docx":
            doc = Document(filepath)
            content = "\n".join(para.text for para in doc.paragraphs)

        else:
            return {"success": False, "error": f"Unsupported file type: {ext}"}

        return {
            "success": True,
            "filepath": filepath,
            "extension": ext,
            "content": content,
            "length": len(content)
        }

    except Exception as e:
        return {"success": False, "error": str(e)}


def list_files(directory: str, extension: str = None) -> list:
    """
    List files in a directory, optionally filtered by extension.
    Returns a list of dicts with file metadata.
    """
    if not os.path.isdir(directory):
        return [{"success": False, "error": f"Directory not found: {directory}"}]

    files_info = []
    for name in os.listdir(directory):
        full_path = os.path.join(directory, name)

        if not os.path.isfile(full_path):
            continue

        if extension and not name.lower().endswith(extension.lower()):
            continue

        stat = os.stat(full_path)
        files_info.append({
            "name": name,
            "size_bytes": stat.st_size,
            "modified": stat.st_mtime
        })

    return files_info


def write_file(filepath: str, content: str) -> dict:
    """
    Write content to a file. Creates parent directories if they don't exist.
    Returns a dict with success/failure status.
    """
    try:
        directory = os.path.dirname(filepath)
        if directory and not os.path.exists(directory):
            os.makedirs(directory, exist_ok=True)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)

        return {
            "success": True,
            "filepath": filepath,
            "bytes_written": len(content.encode("utf-8"))
        }

    except Exception as e:
        return {"success": False, "error": str(e)}


def search_in_file(filepath: str, keyword: str, context_chars: int = 40) -> dict:
    """
    Search for a keyword inside a file's content (case-insensitive).
    Returns matches with surrounding context text.
    """
    file_result = read_file(filepath)

    if not file_result.get("success"):
        return file_result  # propagate the error (file not found, unsupported type, etc.)

    content = file_result["content"]
    lower_content = content.lower()
    lower_keyword = keyword.lower()

    matches = []
    start = 0
    while True:
        idx = lower_content.find(lower_keyword, start)
        if idx == -1:
            break

        context_start = max(0, idx - context_chars)
        context_end = min(len(content), idx + len(keyword) + context_chars)
        snippet = content[context_start:context_end].replace("\n", " ")

        matches.append({
            "position": idx,
            "context": snippet
        })
        start = idx + len(keyword)

    return {
        "success": True,
        "filepath": filepath,
        "keyword": keyword,
        "match_count": len(matches),
        "matches": matches
    }

Overwriting fs_tools.py


In [ ]:
# Cell 5: Test all 4 core tools to confirm fs_tools.py works correctly
#          - read_file on a .txt, .docx, and .pdf resume
#          - list_files on the resumes folder
#          - write_file to create a test output file
#          - search_in_file to find "Python" mentions

import importlib, fs_tools
importlib.reload(fs_tools)
from fs_tools import read_file, list_files, write_file, search_in_file

# Test read_file across formats
print("--- read_file (.txt) ---")
print(read_file("resumes/resume_john_doe.txt"))

print("\n--- read_file (.docx) ---")
print(read_file("resumes/resume_emily_davis.docx"))

print("\n--- read_file (.pdf) ---")
print(read_file("resumes/resume_raj_patel.pdf"))

# Test list_files
print("\n--- list_files (all) ---")
print(list_files("resumes"))

# Test write_file
print("\n--- write_file ---")
print(write_file("output/test_summary.txt", "This is a test summary."))

# Test search_in_file
print("\n--- search_in_file ---")
print(search_in_file("resumes/resume_john_doe.txt", "Python"))

--- read_file (.txt) ---
{'success': True, 'filepath': 'resumes/resume_john_doe.txt', 'extension': '.txt', 'content': 'John Doe\nEmail: john.doe@example.com\nPhone: 555-1234\n\nSummary:\nSoftware engineer with 5 years of experience in Python and cloud infrastructure.\n\nSkills:\nPython, AWS, Docker, SQL\n\nExperience:\nBackend Developer at TechCorp (2020-2025)\n- Built REST APIs using Python and Flask\n- Automated deployments using AWS\n', 'length': 308}

--- read_file (.docx) ---
{'success': True, 'filepath': 'resumes/resume_emily_davis.docx', 'extension': '.docx', 'content': 'Emily Davis\nEmail: emily.davis@example.com\nPhone: 555-5678\n\nSummary:\nBackend engineer with 5 years of experience in Python and Java.\n\nSkills:\nPython, Java, Spring Boot, PostgreSQL\n\nExperience:\nBackend Engineer at FinServe (2020-2025)\n- Built microservices using Python and Spring Boot\n- Optimized PostgreSQL queries', 'length': 315}

--- read_file (.pdf) ---
{'success': True, 'filepath': 'resumes/resu

In [ ]:
# Cell 6: Install the openai package.
# OpenRouter uses an OpenAI-compatible API, so we use this client
# to connect to OpenRouter's models.

!pip install openai -q

In [ ]:
# Cell 7: Connect to OpenRouter using the OpenAI-compatible client.
# The API key is read securely from Colab Secrets (not hardcoded).

from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

print("Client ready:", client is not None)

Client ready: True


In [ ]:
# Cell 8: Define the 4 tools in OpenAI/OpenRouter's function-calling format.
# This schema tells the LLM what each tool does and what parameters it needs,
# so the model can decide which tool(s) to call based on the user's query.

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a resume file (PDF, TXT, or DOCX) and extract its text content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "Path to the file to read, e.g. resumes/resume_john_doe.txt"
                    }
                },
                "required": ["filepath"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory, optionally filtered by file extension.",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {
                        "type": "string",
                        "description": "Directory to list files from, e.g. resumes"
                    },
                    "extension": {
                        "type": "string",
                        "description": "Optional file extension filter, e.g. .pdf or .txt"
                    }
                },
                "required": ["directory"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write text content to a file, creating directories if needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "Path where the file should be written, e.g. output/summary.txt"
                    },
                    "content": {
                        "type": "string",
                        "description": "The text content to write to the file"
                    }
                },
                "required": ["filepath", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_in_file",
            "description": "Search for a keyword inside a file's content (case-insensitive) and return matches with surrounding context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "Path to the file to search in"
                    },
                    "keyword": {
                        "type": "string",
                        "description": "Keyword to search for"
                    }
                },
                "required": ["filepath", "keyword"]
            }
        }
    }
]

print("Number of tools defined:", len(tools_schema))

Number of tools defined: 4


In [ ]:
# Cell 9: Write llm_file_assistant.py — connects the 4 fs_tools functions
# to the LLM via OpenRouter. The run_assistant function lets the model
# call tools across multiple steps (e.g., list files, then search each one)
# before producing a final natural-language answer.

%%writefile llm_file_assistant.py
import json
from openai import OpenAI
from fs_tools import read_file, list_files, write_file, search_in_file

# Map tool names (as strings) to the actual Python functions
AVAILABLE_TOOLS = {
    "read_file": read_file,
    "list_files": list_files,
    "write_file": write_file,
    "search_in_file": search_in_file,
}


def run_assistant(user_query: str, client: OpenAI, tools_schema: list,
                   model: str = "openai/gpt-4o-mini", max_steps: int = 5):
    """
    Send a user query to the LLM, allowing it to call tools across multiple
    steps until it has enough information to produce a final answer.
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant that manages resume files using the available tools. Use tools as many times as needed, step by step, before giving your final answer."},
        {"role": "user", "content": user_query}
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools_schema,
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        tool_calls = response_message.tool_calls

        if not tool_calls:
            return response_message.content

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            print(f"[Step {step + 1}] Tool Call: {function_name}({function_args})")

            if function_name in AVAILABLE_TOOLS:
                result = AVAILABLE_TOOLS[function_name](**function_args)
            else:
                result = {"error": f"Unknown tool: {function_name}"}

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

    return "Max steps reached without a final answer."

Overwriting llm_file_assistant.py


In [ ]:
# Cell 10: Test query 1 - a simple single-tool-call query.
# Expected: LLM calls list_files once, then summarizes the result.

import importlib, llm_file_assistant
importlib.reload(llm_file_assistant)
from llm_file_assistant import run_assistant

answer = run_assistant("List all the files in the resumes folder", client, tools_schema)
print("\nFinal Answer:\n", answer)

[Step 1] Tool Call: list_files({'directory': 'resumes'})

Final Answer:
 Here are the files in the resumes folder:

1. **resume_jane_smith.txt** - Size: 325 bytes
2. **resume_sara_khan.txt** - Size: 312 bytes
3. **resume_emily_davis.docx** - Size: 36,787 bytes
4. **resume_raj_patel.pdf** - Size: 1,645 bytes
5. **resume_john_doe.txt** - Size: 308 bytes
6. **resume_mike_brown.txt** - Size: 290 bytes


In [ ]:
# Cell 11: Test query 2 - a multi-tool-call query.
# Expected: LLM calls list_files once, then search_in_file for each resume,
# then reasons over the results to identify genuine Python experience.

answer = run_assistant(
    "List all resumes, then search each one for the keyword 'Python', and tell me which candidates have genuine Python experience",
    client, tools_schema
)
print("\nFinal Answer:\n", answer)

[Step 1] Tool Call: list_files({'directory': 'resumes', 'extension': '.pdf'})
[Step 2] Tool Call: list_files({'directory': 'resumes', 'extension': '.txt'})
[Step 3] Tool Call: search_in_file({'filepath': 'resumes/resume_raj_patel.pdf', 'keyword': 'Python'})
[Step 3] Tool Call: search_in_file({'filepath': 'resumes/resume_jane_smith.txt', 'keyword': 'Python'})
[Step 3] Tool Call: search_in_file({'filepath': 'resumes/resume_sara_khan.txt', 'keyword': 'Python'})
[Step 3] Tool Call: search_in_file({'filepath': 'resumes/resume_john_doe.txt', 'keyword': 'Python'})
[Step 3] Tool Call: search_in_file({'filepath': 'resumes/resume_mike_brown.txt', 'keyword': 'Python'})

Final Answer:
 Here are the resumes searched for the keyword "Python" and the results regarding their experience:

1. **Raj Patel (resume_raj_patel.pdf)**
   - Matches: 3
   - Experience: 3 years of experience in Python and JavaScript. Built web apps using Python Django.

2. **Jane Smith (resume_jane_smith.txt)**
   - Matches: 3
 

In [ ]:
# Cell 12: Test query 3 - a read + write query.
# Expected: LLM calls read_file to get the resume content, then write_file
# to save a generated summary into the output folder.

answer = run_assistant(
    "Read resume_john_doe.txt in the resumes folder and create a summary file called summary_john_doe.txt in the output folder",
    client, tools_schema
)
print("\nFinal Answer:\n", answer)

# Verify the file was actually created
from fs_tools import read_file
verify = read_file("output/summary_john_doe.txt")
print("\nVerification:\n", verify)

[Step 1] Tool Call: read_file({'filepath': 'resumes/resume_john_doe.txt'})
[Step 2] Tool Call: write_file({'filepath': 'output/summary_john_doe.txt', 'content': 'John Doe\nEmail: john.doe@example.com\nPhone: 555-1234\n\nSummary:\nSoftware engineer with 5 years of experience in Python and cloud infrastructure.\n\nSkills:\nPython, AWS, Docker, SQL\n\nExperience:\nBackend Developer at TechCorp (2020-2025)\n- Built REST APIs using Python and Flask\n- Automated deployments using AWS\n'})

Final Answer:
 I have successfully created the summary file named `summary_john_doe.txt` in the output folder. The file contains the complete content of the resume. If you need any further assistance or modifications, feel free to ask!

Verification:
 {'success': True, 'filepath': 'output/summary_john_doe.txt', 'extension': '.txt', 'content': 'John Doe\nEmail: john.doe@example.com\nPhone: 555-1234\n\nSummary:\nSoftware engineer with 5 years of experience in Python and cloud infrastructure.\n\nSkills:\nPyth

In [ ]:
# Cell 13a: Write requirements.txt listing all dependencies needed to run this project.

%%writefile requirements.txt
openai
pypdf
python-docx
reportlab

Overwriting requirements.txt


In [ ]:
# Cell 13b: Write README.md documenting the project, setup, and usage.

%%writefile README.md
# LLM-Powered File System Assistant

An LLM-powered assistant that manages resume files using tool/function calling.
Built with Python, OpenRouter (OpenAI-compatible API), and file parsing libraries
for PDF, TXT, and DOCX formats.

## Features

**Part A: Core File System Tools** (`fs_tools.py`)
- `read_file(filepath)` — Reads PDF, TXT, or DOCX files and extracts text content
- `list_files(directory, extension=None)` — Lists files in a directory, optionally filtered by extension
- `write_file(filepath, content)` — Writes content to a file, creating directories if needed
- `search_in_file(filepath, keyword)` — Case-insensitive keyword search with surrounding context

**Part B: LLM Integration** (`llm_file_assistant.py`)
- Connects the above tools to an LLM via OpenRouter's function-calling API
- Supports multi-step tool calling — the LLM can call multiple tools in sequence
  (e.g., list files, then search each one) before producing a final answer

## Setup

1. Install dependencies:
```bash
   pip install -r requirements.txt
```

2. Get an OpenRouter API key from [openrouter.ai/keys](https://openrouter.ai/keys)

3. Set your API key as an environment variable:
```bash
   export OPENROUTER_API_KEY="your-key-here"
```
   (In Google Colab, use Colab Secrets instead.)

## Usage

```python
from openai import OpenAI
from llm_file_assistant import run_assistant

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="your-key-here",
)

tools_schema = [...]  # see notebook for full tool schema definitions

answer = run_assistant("List all the files in the resumes folder", client, tools_schema)
print(answer)
```

## Example Queries

- "List all the files in the resumes folder"
- "Find resumes mentioning Python experience"
- "Read resume_john_doe.txt and create a summary file called summary_john_doe.txt in the output folder"

## Project Structure
.
├── fs_tools.py # Core file system tools
├── llm_file_assistant.py # LLM integration + tool-calling loop
├── requirements.txt # Python dependencies
├── resumes/ # Sample resume files (PDF, TXT, DOCX)
├── output/ # Generated output files (summaries, etc.)
└── README.md
## Known Limitations

- `search_in_file` performs simple case-insensitive keyword matching, not semantic
  understanding — e.g., it cannot distinguish "has Python experience" from
  "no Python experience." The LLM layer partially compensates for this by reasoning
  over the raw matches.
- The LLM's tool-calling behavior depends on the underlying model's reasoning quality;
  simpler/cheaper models may occasionally take shortcuts (e.g., assuming a file
  extension without checking).

Overwriting README.md


In [ ]:
# Extra test: matches the brief's exact wording -
# "Read all resumes in the resumes folder"
# Expected: LLM calls list_files, then read_file once per resume,
# then summarizes what it read from each.

answer = run_assistant(
    "Read all resumes in the resumes folder and give me a brief summary of each candidate",
    client, tools_schema
)
print("\nFinal Answer:\n", answer)

[Step 1] Tool Call: list_files({'directory': 'resumes'})
[Step 2] Tool Call: read_file({'filepath': 'resumes/resume_jane_smith.txt'})
[Step 2] Tool Call: read_file({'filepath': 'resumes/resume_sara_khan.txt'})
[Step 2] Tool Call: read_file({'filepath': 'resumes/resume_emily_davis.docx'})
[Step 2] Tool Call: read_file({'filepath': 'resumes/resume_raj_patel.pdf'})
[Step 2] Tool Call: read_file({'filepath': 'resumes/resume_john_doe.txt'})
[Step 2] Tool Call: read_file({'filepath': 'resumes/resume_mike_brown.txt'})

Final Answer:
 Here's a brief summary of each candidate based on their resumes:

1. **Jane Smith**
   - **Email:** jane.smith@example.com
   - **Phone:** 555-2345
   - **Summary:** Data scientist with 6 years of experience in machine learning and Python.
   - **Skills:** Python, TensorFlow, SQL, Pandas
   - **Experience:** Data Scientist at DataCorp (2019-2025); built ML models for fraud detection and worked extensively with SQL databases.

2. **Sara Khan**
   - **Email:** sara